Layer 0: Fetched 2000 records (Total: 2000)...
Layer 0: Fetched 2000 records (Total: 4000)...
Layer 0: Fetched 2000 records (Total: 6000)...
Layer 0: Fetched 2000 records (Total: 8000)...
Layer 0: Fetched 2000 records (Total: 10000)...
Layer 0: Fetched 2000 records (Total: 12000)...
Layer 0: Fetched 2000 records (Total: 14000)...
Layer 0: Fetched 2000 records (Total: 16000)...
Layer 0: Fetched 2000 records (Total: 18000)...
Layer 0: Fetched 2000 records (Total: 20000)...
Layer 0: Fetched 2000 records (Total: 22000)...
Layer 0: Fetched 2000 records (Total: 24000)...
Layer 0: Fetched 2000 records (Total: 26000)...
Layer 0: Fetched 2000 records (Total: 28000)...
Layer 0: Fetched 2000 records (Total: 30000)...
Layer 0: Fetched 2000 records (Total: 32000)...
Layer 0: Fetched 2000 records (Total: 34000)...
Layer 0: Fetched 2000 records (Total: 36000)...
Layer 0: Fetched 2000 records (Total: 38000)...
Layer 0: Fetched 2000 records (Total: 40000)...
Layer 0: Fetched 2000 records (Total: 42000)

In [20]:
import requests
import pandas as pd

# Layer 0 endpoint
url_layer0 = "https://services2.arcgis.com/utBig56dGaw25bwu/ArcGIS/rest/services/MSP_MHSO_Crash_Layer/FeatureServer/0/query"

crashes = []
offset = 0

print("Downloading main crash data (Fatal & Injury)...")

while True:
    payload = {
        "where": "Reporttype IN ('Fatal Crash', 'Injury Crash', 'FATAL CRASH', 'INJURY CRASH')",
        "outFields": "*",
        "f": "json",
        "returnGeometry": "true",
        "outSR": "4326",
        "resultOffset": str(offset),
        "resultRecordCount": "2000",
        "orderByFields": "OBJECTID ASC",
    }
    
    response = requests.post(url_layer0, data=payload)
    res = response.json()
    
    if "error" in res:
        print("ArcGIS Error:", res["error"])
        break
        
    features = res.get("features", [])
    if not features:
        break
        
    for feat in features:
        row = feat.get("attributes", {})
        geom = feat.get("geometry", {})
        if geom and "x" in geom and "y" in geom:
            row["longitude"] = geom["x"]
            row["latitude"] = geom["y"]
        crashes.append(row)
        
    print(f"Layer 0: Fetched {len(features)} records (Total: {len(crashes)})...")
    
    if len(features) < 2000:
        break
    offset += len(features)
df_crashes = pd.DataFrame(crashes)
df_crashes.to_csv("msp_crashes_main_filtered.csv", index=False)
print(f"Saved {len(df_crashes)} filtered records to msp_crashes_main_filtered.csv")

# Extract unique Reportnumber values for subtable query
report_ids = [str(x) for x in df_crashes["Reportnumber"].dropna().unique() if str(x).strip()]
print(f"Extracted {len(report_ids)} unique Reportnumber values.\n")

subtables = {
    "vehicles": "1",
    "passengers": "2",
    "nonmotorists": "3",
    "drivers": "4",
}

chunk_size = 2000

for name, layer_id in subtables.items():
    print(f"Starting {name} (Layer {layer_id})...")
    sub_records = []
    
    for i in range(0, len(report_ids), chunk_size):
        chunk = report_ids[i : i + chunk_size]
        formatted_ids = ", ".join([f"'{cid}'" for cid in chunk])
        where_clause = f"Reportnumber IN ({formatted_ids})"
        
        offset = 0
        while True:
            sub_url = f"https://services2.arcgis.com/utBig56dGaw25bwu/ArcGIS/rest/services/MSP_MHSO_Crash_Layer/FeatureServer/{layer_id}/query"
            
            sub_payload = {
                "where": where_clause,
                "outFields": "*",
                "f": "json",
                "resultOffset": str(offset),
                "resultRecordCount": "2000",
                "orderByFields": "OBJECTID ASC",
            }
            
            res = requests.post(sub_url, data=sub_payload).json()
            features = res.get("features", [])
            if not features:
                break
                
            for feat in features:
                sub_records.append(feat.get("attributes", {}))
                
            if len(features) < 2000:
                break
            offset += len(features)

    df_sub = pd.DataFrame(sub_records)
    df_sub.to_csv(f"msp_{name}_filtered.csv", index=False)
    print(f"Saved {len(df_sub)} records to msp_{name}_filtered.csv\n")

print("Finished downloading all crash and subtable data.")

Starting vehicles (Layer 1)...
Saved 341710 records to msp_vehicles_filtered.csv

Starting passengers (Layer 2)...
Saved 137606 records to msp_passengers_filtered.csv

Starting nonmotorists (Layer 3)...
Saved 21937 records to msp_nonmotorists_filtered.csv

Starting drivers (Layer 4)...
Saved 329521 records to msp_drivers_filtered.csv

Starting circumstances (Layer 5)...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [50]:
crashes = pd.read_csv("msp_crashes_main_filtered.csv")
vehicles = pd.read_csv("msp_vehicles_filtered.csv")
nonmotorists = pd.read_csv("msp_nonmotorists_filtered.csv")
drivers = pd.read_csv("msp_drivers_filtered.csv")
passengers = pd.read_csv("msp_passengers_filtered.csv")
severe_nonmo = nonmotorists[nonmotorists['NonMotoristInjurySeverity'].isin(["Suspected Serious Injury", "Fatal Injury"])]
severe_dri = drivers[drivers['DriverInjurySeverity'].isin(["Suspected Serious Injury", "Fatal Injury"])]
severe_pass = passengers[passengers['PassengerInjurySeverity'].isin(["Suspected Serious Injury", "Fatal Injury"])]
severe_crash_set = set(list(severe_nonmo['Reportnumber'].unique()) + list(severe_pass['Reportnumber'].unique()) + list(severe_dri['Reportnumber'].unique()))

/var/folders/0r/74g1680s6rz9r8mbc5tx0_0c0000gn/T/ipykernel_27667/167780742.py:1: DtypeWarning: Columns (0: MaintenanceLocation, 1: MaintenanceWorkersPresent, 2: Offroaddescription, 3: Trafficcontrolfunctioning) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv("msp_crashes_main_filtered.csv")


In [55]:
severe_crashes = crashes[crashes['Reportnumber'].isin(severe_crash_set)]
severe_vehicles = vehicles[vehicles['Reportnumber'].isin(severe_crash_set)]
severe_drivers = drivers[drivers['Reportnumber'].isin(severe_crash_set)]
severe_passengers = passengers[passengers['Reportnumber'].isin(severe_crash_set)]
severe_nonmotorists = nonmotorists[nonmotorists['Reportnumber'].isin(severe_crash_set)]
severe_crashes.to_csv("crashes.csv")
severe_vehicles.to_csv("vehicles.csv")
severe_drivers.to_csv("drivers.csv")
severe_passengers.to_csv("passengers.csv")
severe_nonmotorists.to_csv("nonmotorists.csv")

In [61]:
crashes['Harmfuleventone'].unique()

<StringArray>
[                  'Fixed Object',                  'Other Vehicle',
                     'Pedestrian',                        'Bicycle',
            'Other Non-Collision',                'Units Separated',
                        'Offroad',                          'Other',
                 'Parked Vehicle',              'Explosion or Fire',
                       'Overturn',                              nan,
                        'Unknown',                         'Animal',
                'Downhil Runaway',               'Other Conveyance',
 'Fell Jumped from Motor Vehicle',                   'Other Object',
                       'Jacknife',               'Other Pedalcycle',
                        'Backing',                  'Spilled Cargo',
                      'Immersion',       'Thrown or Falling Object',
                  'Railway Train',                         'U-Turn']
Length: 26, dtype: str

In [62]:
drivers

,OBJECTID,Reportnumber,DriverID,DriverAirbagDeployed,DriverAlcoholTestIndicator,DriverAlcoholTestType,DriverBAC,DriverCondition,Driverdistractedby,DriverDOB,DriverDrugTestIndicator,DriverDrugTestResult,DriverEjection,DriverEquipmentProblem,DriverHasCDL,DriverInjurySeverity,DriverSafetyEquipment,DriverSex,DriverSubstanceUse,DriverVehicleID,DriverAge,DriverMature,DriverYoung,DriverDistracted,DriverSpeeding,DriverAggressive,DriverImpaired,OnMotorcycle,Unbelted,Description
0,147,AE58770014,a5d9384d-0427-478e-aaf7-4bf4b48717f7,NaN,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,N,No Apparent Injury,NaN,NaN,NaN,e382f565-02b3-46be-9cd9-043f9a372dda,NaN,0,0,0,0,0,0,0,0,","
1,420,CB6036002P,8150588e-b652-478a-9ece-822997452a09,NaN,NaN,NaN,NaN,NaN,Not Distracted,NaN,NaN,NaN,NaN,Unknown,N,No Apparent Injury,Unknown,NaN,NaN,c0ee6ba9-5e6b-4642-a61f-e89df933912d,NaN,0,0,0,0,0,0,0,0,","
2,590,DV0140001T,b6e8d1eb-ed72-4fdb-8fe6-0088762757b4,Not Deployed,NaN,NaN,NaN,NaN,Not Distracted,NaN,NaN,NaN,NaN,NaN,N,No Apparent Injury,NaN,NaN,NaN,c2b20de2-c49e-4d23-ad48-39da6b6f45aa,NaN,0,0,0,0,0,0,0,0,","
3,624,MCP23530066,07209136-59fd-42ea-a209-6ded49b9ac91,Deployed - Unknown,NaN,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,Unknown,N,No Apparent Injury,Unknown,NaN,Unknown,a11fb75a-f6ab-4911-a624-9f282860ad87,NaN,0,0,1,1,1,0,0,0,","
4,627,MSP4857001S,16af3bd7-a7a2-4011-a683-353eb031b3c9,Deployed - Unknown,NaN,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,Unknown,N,No Apparent Injury,Unknown,NaN,Unknown,b65f6b41-e44f-40e4-86b0-00a10e00ee2a,NaN,0,0,0,0,0,0,0,0,","
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329516,1111908,AE4633008N,399b8fa3-0801-4137-9b35-4607b7521556,NaN,NaN,NaN,NaN,Apparently Normal,Unknown,1994-09-06,NaN,NaN,Fully Ejected,No Misuse,N,Suspected Serious Injury,Helmet and Eye Protection,M,None Detected,e3a0f30d-88b9-40a7-943f-1e3db58393ac,28.676838,0,0,0,0,0,0,1,0,"M, 28"
329517,1111916,AE55830068,0f5f6c0e-aa33-45d4-8cdc-9608e168022e,NaN,NaN,NaN,NaN,Apparently Normal,Unknown,1995-12-29,NaN,NaN,Partially Ejected,NaN,N,Suspected Serious Injury,NaN,M,None Detected,ebf2a68f-d87d-49f0-9765-93fb6ba1b714,27.083376,0,0,1,0,0,0,0,1,"M, 27"
329518,1111920,ADK4560041,a63c8a80-4357-428b-a4fb-c07b2b9a48f6,Deployed - Front,NaN,NaN,NaN,Apparently Normal,Not Distracted,1976-01-22,NaN,NaN,Trapped,No Misuse,N,Suspected Serious Injury,Unknown,M,NaN,088b9df5-b109-468a-94f6-e645739f5e8e,47.636844,0,0,0,0,0,0,0,0,"M, 47"
329519,1112150,AE6463001W,bcaab845-713d-4d78-8d2a-a342628a451b,Not Deployed,NaN,NaN,NaN,NaN,Looked But Did Not See,1982-02-22,NaN,NaN,Not Ejected/Trapped,NaN,N,No Apparent Injury,NaN,U,NaN,e1b4ba3b-ac89-4b48-96cf-c4d37f480fc6,41.035750,0,0,0,0,0,0,0,0,"U, 41"


In [63]:
vehicles

,OBJECTID,Reportnumber,Vehicleid,CommercialBodyType,CommercialCarrierClassification,CommercialConfiguration,CommercialGVW,CommercialHazmatClass,CommercialHazmatNumber,CommercialHazmatPlacardVisible,CommercialHazmatSpill,CommercialNumberofaxles,Continuedirection,Damageextent,Driverlessvehicle,Emergencymotorvehicleuse,Fire,Firstimpact,Goingdirection,Mainimpact,Mostharmfulevent,Parkedvehicle,Speedlimit,Towedunittype,Unitnumber,Vehiclebodytype,VehicleBusUse,VehicleLeftScene,Vehiclemake,Vehiclemodel,Vehiclemovement,VehicleSpecialFunction,Vehicleyear,VIN,Description
0,96,DA4053003X,f7c8b9d5-e61b-44d4-8d1e-45912d3b6807,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,Functional,N,N,NaN,12 O'Clock,E,12 O'Clock,Other Vehicle,N,30.0,NaN,1.0,Passenger Car,NaN,N,TOYOTA,CAMRY,1.0,NaN,2003.0,JTDBE32K430181702,2003 TOYOTA CAMRY
1,107,ZU4251004B,73fff59f-7283-4b7f-ad34-5abd281b819e,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,Functional,N,N,NaN,1 O'Clock,E,1 O'Clock,Other Vehicle,N,45.0,NaN,1.0,Passenger Car,NaN,N,HONDA,CIVIC,7.0,NaN,2015.0,19XFB2E59FE066258,2015 HONDA CIVIC
2,218,DA3756002T,d481134a-0240-4ada-982b-12c1279923b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,Superficial,N,N,NaN,12 O'Clock,E,12 O'Clock,Other Vehicle,N,40.0,NaN,1.0,Passenger Car,NaN,N,HONDA,4S,1.0,NaN,2012.0,1HGCP2F38CA166224,2012 HONDA 4S
3,359,AC1152002T,a0246f70-1c42-4c2b-bc5d-3599abee5cb5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S,Disabling,N,N,NaN,12 O'Clock,S,12 O'Clock,Other Vehicle,N,45.0,NaN,1.0,Passenger Car,NaN,N,HONDA,CIVIC,3.0,NaN,2010.0,2HGFA1F56AH32963,2010 HONDA CIVIC
4,429,AE5645009Q,d337b604-6adf-4bf8-a20f-99edb538c236,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,Superficial,N,N,NaN,4 O'Clock,E,4 O'Clock,Other Vehicle,N,25.0,NaN,2.0,Passenger Car,NaN,N,HOND,ACCORD,1.0,NaN,2016.0,1HGCT2B06GA003391,2016 HOND ACCORD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341705,1222830,HB03560003,b6d90b81-daad-4658-90e1-08a9dd349f23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,No Damage,N,N,N,12 O'Clock,E,12 O'Clock,Pedestrian,N,10.0,NaN,1.0,Passenger Car,NaN,Y,MITS,4S,5.0,NaN,2018.0,JA4AP3AW3JZ009139,2018 MITS 4S
341706,1222850,DS0421001R,fd7824aa-9e1f-4534-8ade-f905a6281fc3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,No Damage,N,N,N,6 O'Clock,N,6 O'Clock,Pedestrian,N,5.0,NaN,1.0,Truck - Pickup,NaN,Y,FORD,F150,11.0,Work Zone Equipment,2014.0,1FTMF1EM3EKG51782,2014 FORD F150
341707,1222873,MCP3053006J,74f6f087-1cc5-4adc-87dc-f488f3371f69,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,U,No Damage,N,N,N,Unknown,U,Unknown,Pedestrian,N,15.0,NaN,1.0,Passenger Car,NaN,Y,CHRY,4D,11.0,NaN,2013.0,1C3CCBCG2DN522697,2013 CHRY 4D
341708,1222881,ADJ865003D,1a7c2532-bf86-45ae-b411-1c12d5fd0932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,W,No Damage,N,N,N,11 O'Clock,W,11 O'Clock,Pedestrian,N,5.0,NaN,1.0,Passenger Car,NaN,Y,TOYOTA,RAV4,13.0,NaN,2018.0,JTMZFREV1JD110645,2018 TOYOTA RAV4


In [64]:
drivers[drivers['Reportnumber']=="DA4053003X"]

,OBJECTID,Reportnumber,DriverID,DriverAirbagDeployed,DriverAlcoholTestIndicator,DriverAlcoholTestType,DriverBAC,DriverCondition,Driverdistractedby,DriverDOB,DriverDrugTestIndicator,DriverDrugTestResult,DriverEjection,DriverEquipmentProblem,DriverHasCDL,DriverInjurySeverity,DriverSafetyEquipment,DriverSex,DriverSubstanceUse,DriverVehicleID,DriverAge,DriverMature,DriverYoung,DriverDistracted,DriverSpeeding,DriverAggressive,DriverImpaired,OnMotorcycle,Unbelted,Description
111,17476,DA4053003X,00a473b2-1a4a-4760-920a-4c980cbdc77c,NaN,NaN,NaN,NaN,NaN,Not Distracted,1977-04-07,NaN,NaN,NaN,NaN,N,No Apparent Injury,NaN,F,NaN,d9b0621f-f1d7-46bf-9a70-eb0afea35865,41.490243,0,0,0,0,0,0,0,0,"F, 41"
822,73392,DA4053003X,99a34a6b-8399-40df-a4db-62feb2f56a3a,NaN,NaN,NaN,NaN,Apparently Normal,Not Distracted,1978-01-25,NaN,NaN,NaN,NaN,N,Possible Injury,Shoulder/Lap Belt,F,NaN,f7c8b9d5-e61b-44d4-8d1e-45912d3b6807,40.688036,0,0,0,0,0,0,0,0,"F, 40"


In [65]:
vehicles[vehicles['Reportnumber']=="DA4053003X"]

,OBJECTID,Reportnumber,Vehicleid,CommercialBodyType,CommercialCarrierClassification,CommercialConfiguration,CommercialGVW,CommercialHazmatClass,CommercialHazmatNumber,CommercialHazmatPlacardVisible,CommercialHazmatSpill,CommercialNumberofaxles,Continuedirection,Damageextent,Driverlessvehicle,Emergencymotorvehicleuse,Fire,Firstimpact,Goingdirection,Mainimpact,Mostharmfulevent,Parkedvehicle,Speedlimit,Towedunittype,Unitnumber,Vehiclebodytype,VehicleBusUse,VehicleLeftScene,Vehiclemake,Vehiclemodel,Vehiclemovement,VehicleSpecialFunction,Vehicleyear,VIN,Description
0,96,DA4053003X,f7c8b9d5-e61b-44d4-8d1e-45912d3b6807,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,Functional,N,N,NaN,12 O'Clock,E,12 O'Clock,Other Vehicle,N,30.0,NaN,1.0,Passenger Car,NaN,N,TOYOTA,CAMRY,1.0,NaN,2003.0,JTDBE32K430181702,2003 TOYOTA CAMRY
216,9702,DA4053003X,d9b0621f-f1d7-46bf-9a70-eb0afea35865,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,E,Superficial,N,N,N,6 O'Clock,E,6 O'Clock,Other Vehicle,N,25.0,NaN,2.0,Passenger Car,NaN,N,NISSAN,ROUGE,3.0,NaN,2015.0,5N1AT2MVXFC769119,2015 NISSAN ROUGE


In [67]:
passengers[passengers['Reportnumber']=="DA4053003X"]

,OBJECTID,Reportnumber,PassengerID,PassengerAirbagDeployed,PassengerDOB,PassengerEjection,PassengerEquipmentProblem,PassengerInjurySeverity,PassengerSafetyEquipment,PassengerSeat,PassengerSeatingLocation,PassengerSeatingRow,PassengerSex,PassengerVehicleID,PassengerAge,OnMotorcycle,Unbelted,Description
1462,65036,DA4053003X,86150df1-77a0-4fe0-bee7-ad944b825230,NaN,2002-10-30,NaN,NaN,Possible Injury,Shoulder/Lap Belt,Right,Right Front Seat,1.0,M,f7c8b9d5-e61b-44d4-8d1e-45912d3b6807,15.926405,0,0,"M, 15"


In [69]:
nonmotorists

,OBJECTID,Reportnumber,NonMotoristID,NonMotoristActions,NonMotoristAlcoholTestindicator,NonMotoristAlcoholTestType,NonMotoristBAC,NonMotoristCondition,NonMotoristDOB,NonMotoristDrugTestIndicator,NonMotoristDrugTestResult,NonMotoristInjurySeverity,NonMotoristLocation,NonMotoristMovement,NonMotoristObeyTrafficSignal,NonMotoristSafetyEquipment,NonMotoristSex,NonMotoristSubstanceUse,NonMotoristType,NonMotoristUnitNumberFirstStrik,NonMotoristVisibility,NonMotoristAge,NonMotoristImpaired,Pedestrian,Bicyclist,Description
0,1,ADJ8180010,0fc96d0f-024f-45be-8b67-4fc898b0fb6d,In Roadway Improperly,NaN,NaN,NaN,NaN,1998-03-21,NaN,NaN,Possible Injury,On Roadway Not at Crosswalk,Cross/Enter at Intersection,No Pedestrian Signal,NaN,M,None Detected,Pedestrian,1.0,Light Clothing,19.784116,0,1,0,"Pedestrian - M, 19"
1,2,AE4883009R,812cae61-4231-436d-8273-45f0d0b3e49e,NaN,NaN,NaN,NaN,NaN,2017-01-02,NaN,NaN,No Apparent Injury,NaN,NaN,NaN,NaN,F,NaN,Other,1.0,NaN,0.999336,0,0,0,"Other - F, 0"
2,3,MSP5924008L,9def17df-93fc-4ce3-8114-06d1ff909793,No Improper Actions,NaN,NaN,NaN,Apparently Normal,1957-10-02,NaN,NaN,Possible Injury,Shoulder,Walking/Riding With Traffic,NaN,NaN,M,None Detected,Bicyclist,1.0,Light Clothing,60.253120,0,0,1,"Bicyclist - M, 60"
3,4,AC2019001R,62073ff4-3f54-4a82-b6e7-8ca08a86fb6c,Other,NaN,NaN,NaN,Other Handicap,1962-01-09,NaN,NaN,Possible Injury,Other,Other,Other,NaN,M,None Detected,Pedestrian,1.0,Light Clothing,55.981985,0,1,0,"Pedestrian - M, 55"
4,5,CB5047004M,4e3aad20-24ed-41b3-bc3c-6c4787f65f46,Disabled Vehicle Related,NaN,NaN,NaN,Apparently Normal,1960-09-30,NaN,NaN,Possible Injury,Shoulder,Push/Work on Vehicle,NaN,NaN,M,None Detected,Pedestrian,2.0,NaN,57.263325,0,1,0,"Pedestrian - M, 57"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21932,25646,AE66750004,05552e6d-cc17-4c92-acfb-b4a7d76326b0,Dart Dash,NaN,NaN,NaN,Apparently Normal,2009-06-22,NaN,NaN,Possible Injury,In School Bus Zone,Getting Off/On Vehicle,No Pedestrian Signal,NaN,M,NaN,Pedestrian,1.0,Mixed Clothing,13.820955,0,1,0,"Pedestrian - M, 13"
21933,25647,AC2178009L,2ce4a363-f97f-4fc7-9f9f-bc904119b785,No Improper Actions,NaN,NaN,NaN,Apparently Normal,1935-10-08,NaN,NaN,Possible Injury,Inside Building,Other,No Pedestrian Signal,NaN,M,None Detected,Other,NaN,Light Clothing,87.289951,0,0,0,"Other - M, 87"
21934,25648,MCP94290007,57b5030d-9625-4bda-96dd-e1f12fc8c0bf,In Roadway Improperly,NaN,NaN,NaN,Apparently Normal,1991-01-27,NaN,NaN,Possible Injury,On Roadway Not at Crosswalk,NaN,NaN,NaN,M,NaN,Pedestrian,NaN,Light Clothing,32.132077,0,1,0,"Pedestrian - M, 32"
21935,25669,ADJ39200BS,27920bd1-6fd0-417a-9b2a-5f07a7c16aea,NaN,NaN,NaN,NaN,Apparently Normal,1978-04-16,NaN,NaN,Suspected Minor Injury,Other,Other Working,Other,Reflective Clothing,M,NaN,Pedestrian,1.0,Light Clothing,44.781207,0,1,0,"Pedestrian - M, 44"


In [73]:
vehicles[pd.isnull(vehicles['Unitnumber'])]

,OBJECTID,Reportnumber,Vehicleid,CommercialBodyType,CommercialCarrierClassification,CommercialConfiguration,CommercialGVW,CommercialHazmatClass,CommercialHazmatNumber,CommercialHazmatPlacardVisible,CommercialHazmatSpill,CommercialNumberofaxles,Continuedirection,Damageextent,Driverlessvehicle,Emergencymotorvehicleuse,Fire,Firstimpact,Goingdirection,Mainimpact,Mostharmfulevent,Parkedvehicle,Speedlimit,Towedunittype,Unitnumber,Vehiclebodytype,VehicleBusUse,VehicleLeftScene,Vehiclemake,Vehiclemodel,Vehiclemovement,VehicleSpecialFunction,Vehicleyear,VIN,Description
2763,138428,A0058343,8800877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,N,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,NaN,NaN,
2764,138429,A0061888,8808883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,N,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,NaN,NaN,
2765,138430,A0062654,8810925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,N,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,NaN,NaN,
2766,138431,A0064458,8815451,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,N,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,NaN,NaN,
2767,138432,A0066465,8821585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,N,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,NaN,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233992,944412,A0094056,8861416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,U,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,0.0,NaN,0
233993,944413,A0134452,8934551,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,U,N,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,0.0,NaN,0
233994,944415,A0082454,8850242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Disabling,N,U,Y,Unknown,0,Unknown,Fixed Object,N,55.0,Unknown,NaN,Passenger Car,NaN,N,NISSAN,SENTRA,99.0,Unknown,2017.0,3N1AB7AP3HY343747,2017 NISSAN SENTRA
233995,944416,A0134452,8934550,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Unknown,N,U,Y,Unknown,9,Unknown,Unknown,N,NaN,Unknown,NaN,Unknown,NaN,N,NaN,NaN,99.0,Unknown,0.0,NaN,0
